# Project 5 — Local Cover Letter Generator

## Generate Tailored Cover Letters from JD + Resume

**Goal:** Combine a job description and resume into a polished, tailored
cover letter — learning multi-input prompting and context synthesis.

**Stack:** Ollama · LangChain · Jupyter

### What You'll Learn

1. **Multi-input prompting** — blending JD and resume into one prompt
2. **Context extraction** — pulling key requirements and candidate strengths
3. **Tone control** — adjusting formality, enthusiasm, and length
4. **Variant generation** — creating multiple letter styles for comparison
5. **Quality critique** — self-evaluation of generated letters

### Prerequisites

- Ollama running with `qwen3:8b` pulled

In [ ]:
# Install dependencies (uncomment and run once)
# !pip install -q langchain langchain-ollama langchain-core

## Step 1 — Verify Ollama and Configure LLM

In [ ]:
import requests
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

try:
    r = requests.get("http://localhost:11434/api/tags", timeout=5)
    print(f"Ollama running — {len(r.json().get('models', []))} models")
except: print("Start Ollama first!")

llm = ChatOllama(model="qwen3:8b", temperature=0.4)
print("LLM configured")

## Step 2 — Sample Job Description and Resume

We provide both inputs that will be synthesized into a cover letter.

In [ ]:
job_description = """
Senior Machine Learning Engineer — DataFlow Inc.

About the Role:
We're looking for an experienced ML engineer to lead our recommendation
system team. You'll design and deploy models that serve millions of users.

Requirements:
- 4+ years in ML engineering or applied data science
- Strong Python skills and experience with PyTorch or TensorFlow
- Experience deploying ML models to production at scale
- Knowledge of recommendation systems, NLP, or computer vision
- Experience with cloud platforms (AWS, GCP, or Azure)
- Excellent communication skills for cross-team collaboration
- Experience mentoring junior engineers is a plus

What We Offer:
- Competitive salary and equity
- Flexible remote work policy
- Learning and conference budget
"""

resume_text = """
Jane Chen — ML Engineer
Email: jane.chen@email.com | GitHub: github.com/janechen

EXPERIENCE:
Machine Learning Engineer, TechStartup Inc. (2021-Present)
- Built and deployed a product recommendation engine serving 2M+ daily users
- Reduced model inference latency by 40% through model optimization and caching
- Led migration from TensorFlow to PyTorch, improving training speed by 2x
- Mentored 2 junior engineers on ML best practices and code review

Data Scientist, Analytics Corp (2019-2021)
- Developed NLP pipeline for customer feedback classification (92% accuracy)
- Built A/B testing framework for model comparison, adopted by 3 teams
- Deployed models on AWS SageMaker with CI/CD pipelines

SKILLS: Python, PyTorch, TensorFlow, scikit-learn, SQL, AWS, Docker, Git
EDUCATION: M.S. Computer Science, State University (2019)
"""

print(f"JD: {len(job_description.split())} words")
print(f"Resume: {len(resume_text.split())} words")

## Step 3 — Extract Key Requirements and Strengths

Before generating the letter, let's explicitly extract what the JD needs
and what the candidate offers. This intermediate step improves letter quality.

In [ ]:
extract_prompt = ChatPromptTemplate.from_messages([
    ("system", """Analyze the job description and resume. Extract:

1. **Top 5 JD Requirements** — the most important skills/experiences needed
2. **Candidate's Matching Strengths** — where the resume directly matches
3. **Gaps** — JD requirements not clearly addressed in the resume
4. **Unique Value** — candidate strengths beyond what's required

Be specific and concise."""),
    ("human", "Job Description:\n{jd}\n\nResume:\n{resume}"),
])

analysis = (extract_prompt | llm | StrOutputParser()).invoke({
    "jd": job_description,
    "resume": resume_text,
})

print("=== JD-Resume Analysis ===")
print(analysis)

## Step 4 — Generate the Cover Letter

Now we generate a complete cover letter that synthesizes both inputs,
highlighting the candidate's relevant experience and enthusiasm.

In [ ]:
cover_letter_prompt = ChatPromptTemplate.from_messages([
    ("system", """Write a professional cover letter for this job application.

Guidelines:
- Open with enthusiasm for the specific role and company
- Highlight 2-3 most relevant experiences that match JD requirements
- Use specific metrics and achievements from the resume
- Show knowledge of what the company does
- Close with a clear call to action
- Keep it to 3-4 paragraphs, under 400 words
- Tone: confident but not arrogant, professional but personable"""),
    ("human", "Job Description:\n{jd}\n\nResume:\n{resume}"),
])

cover_letter = (cover_letter_prompt | llm | StrOutputParser()).invoke({
    "jd": job_description,
    "resume": resume_text,
})

print("=== Generated Cover Letter ===")
print(cover_letter)

## Step 5 — Generate Tone Variants

Different companies prefer different communication styles.
Let's generate variants to see how tone affects the letter.

In [ ]:
tone_variants = {
    "formal": "Write in a highly formal, traditional business letter style.",
    "conversational": "Write in a warm, conversational but professional tone.",
    "technical": "Write in a technical style, emphasizing engineering depth and specific technologies.",
}

for tone_name, instruction in tone_variants.items():
    variant_prompt = ChatPromptTemplate.from_messages([
        ("system", f"Write a cover letter for this job application. {instruction} Keep it under 300 words."),
        ("human", "Job Description:\n{jd}\n\nResume:\n{resume}"),
    ])
    result = (variant_prompt | llm | StrOutputParser()).invoke({
        "jd": job_description,
        "resume": resume_text,
    })
    print(f"\n{'='*60}")
    print(f"Tone: {tone_name}")
    print(f"{'='*60}")
    print(result[:500])
    print("..." if len(result) > 500 else "")

## Step 6 — Self-Critique the Letter

Let's use the LLM as a **reviewer** to evaluate its own output.
This teaches the pattern of generation → evaluation → revision.

In [ ]:
critique_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a hiring manager reviewing this cover letter for the given job.

Rate each aspect from 1-10 and explain briefly:
1. **Relevance** — Does it address the job requirements?
2. **Specificity** — Does it use concrete examples and metrics?
3. **Tone** — Is it appropriate for the role and company?
4. **Persuasiveness** — Would you want to interview this candidate?
5. **Length** — Is it the right length?

Then give one specific suggestion for improvement."""),
    ("human", "Job Description:\n{jd}\n\nCover Letter:\n{letter}"),
])

critique = (critique_prompt | llm | StrOutputParser()).invoke({
    "jd": job_description,
    "letter": cover_letter,
})

print("=== Cover Letter Critique ===")
print(critique)

## Limitations & Tradeoffs

| Aspect | What happens | How to improve |
|--------|-------------|----------------|
| **Generic phrasing** | Model may produce cliché language | Provide more specific company context |
| **Company knowledge** | Model doesn't know real company details | Add company research as extra input |
| **Fabrication** | May embellish beyond what's in the resume | Cross-check against actual resume |
| **Format variation** | Different runs produce different structures | Use a template with fill-in-the-blank |
| **Length control** | May be too long or too short | Specify word count in prompt |

### What this project does NOT cover
- PDF formatting and export
- ATS keyword optimization
- Multiple application tracking
- Company-specific research integration

## What You Learned

1. **Multi-input prompting** — blending two text sources into one coherent output
2. **Context extraction** — analyzing JD and resume before generation
3. **Tone variants** — controlling style through prompt instructions
4. **Self-critique** — using the model to evaluate its own output
5. **Quality assessment** — structured evaluation with scoring rubrics

## Exercises

1. **Use your own resume + a real JD** — generate a letter for a role you're interested in
2. **Add company research** — include a paragraph about the company and see if the letter improves
3. **Iterative refinement** — take the critique feedback and regenerate an improved version
4. **A/B test** — have a friend rate two letter variants without knowing which is which

---

*Next project: **06 — Local Email Reply Assistant** (intent classification + reply drafting)*